# Глава 3. Предобработка данных — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Первый осмотр: что не так с данными

Данные из реальных систем почти всегда грязные: пропуски, разные написания одного и того же, числа в виде текста, повторы. Прежде чем что-то считать, данные осматривают по короткому чек-листу. Так вы найдёте проблемы до того, как они испортят отчёт.

Теория — на сайте курса: `ch03/01-inspect.html`. Задачи «собери код из строк» (1) остались на сайте.

### Размер выгрузки · разминка

Сохраните число строк в `n_rows`, а число столбцов — в `n_cols`.

*Подсказки (2): `dsc.hint('ch03/01-inspect/shape')`*

In [ ]:
%%task ch03/01-inspect/shape
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Категории и количества · разминка

Разложите столбцы `orders_3m.csv` на два списка названий:

- `quantitative` — количественные: те, где осмысленны сумма и среднее;
- `categorical` — категориальные: признаки, по которым гостей и заказы делят на группы.

Номер заказа, номер карты и дату-время не включайте ни в один список: это идентификаторы и дата. Порядок названий в списках не важен.

*Подсказки (3): `dsc.hint('var-kind')`*

In [ ]:
%%task ch03/01-inspect/var-kind
import pandas as pd
orders = pd.read_csv("orders_3m.csv")
print(orders.dtypes)
print(orders.nunique())
quantitative = []
categorical = []

### Где больше всего пропусков · разминка

Посчитайте `missing` — число пропусков по каждому столбцу (Series), и сохраните в `worst` название столбца, где пропусков больше всего.

*Подсказки (2): `dsc.hint('worst')`*

In [ ]:
%%task ch03/01-inspect/worst
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Отчёт о пропусках · база

Сохраните в `miss_pct` долю пропусков по каждому столбцу **в процентах**, округлённую до одного знака и отсортированную по убыванию. Столбцы без пропусков тоже должны остаться в отчёте.

*Подсказки (2): `dsc.hint('ch03/01-inspect/miss-share')`*

In [ ]:
%%task ch03/01-inspect/miss-share
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Сколько вариантов у города · база

Сколько разных написаний города в выгрузке? Сохраните это число (без учёта пропусков) в `n_variants`, а в `top_city` — самое частое написание.

*Подсказки (2): `dsc.hint('variants')`*

In [ ]:
%%task ch03/01-inspect/variants
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Повторы · база

Сохраните:

- `n_full` — число полных дублей строк;
- `ids_ok` — `True`, если номера карт уникальны, иначе `False`;
- `n_extra` — на сколько строк в таблице больше, чем разных номеров карт.

*Подсказки (2): `dsc.hint('dups')`*

In [ ]:
%%task ch03/01-inspect/dups
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Что мешает бонусам · база

Сколько **непустых** значений `bonus` не состоят только из цифр? Сохраните в `n_bad`. В `bad_values` сохраните список этих значений без повторов, отсортированный по алфавиту.

*Подсказки (2): `dsc.hint('not-digits')`*

In [ ]:
%%task ch03/01-inspect/not-digits
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Функция осмотра · посложнее

Напишите функцию `overview(df)`, которая возвращает DataFrame с одной строкой на каждый столбец исходной таблицы (индекс — названия столбцов) и тремя столбцами:

- `missing` — число пропусков;
- `missing_pct` — доля пропусков в процентах, округлённая до одного знака;
- `unique` — число разных значений.

Функция пригодится для любой новой выгрузки.

*Подсказки (2): `dsc.hint('overview')`*

In [ ]:
%%task ch03/01-inspect/overview
import pandas as pd
df = pd.read_csv("clients_raw.csv")

def overview(df):
    ...

overview(df)

### Лишние пробелы · со звёздочкой

В каких столбцах есть значения с пробелами в начале или в конце? Сохраните в `with_spaces` список названий таких столбцов в порядке, как они идут в таблице.

*Подсказки (2): `dsc.hint('ch03/01-inspect/spaces')` · задача по желанию*

In [ ]:
%%task ch03/01-inspect/spaces
import pandas as pd
df = pd.read_csv("clients_raw.csv")

## Урок 2. Пропуски

Пустые ячейки есть почти в любых данных. Разбираемся, как их найти (в том числе замаскированные под «не указано»), чем они опасны для расчётов и что с ними делать: удалить, заполнить или оставить.

Теория — на сайте курса: `ch03/02-missing.html`. Задачи «собери код из строк» (1) остались на сайте.

### Анкеты без города · разминка

Сохраните в `n_no_city` число анкет, в которых не указан город.

*Подсказки (2): `dsc.hint('no-city')`*

In [ ]:
%%task ch03/02-missing/no-city
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Скрытые пропуски в дате рождения · разминка

Загрузите файл так, чтобы «не указано» стало пропуском, и сохраните в `n_no_birth` число анкет без даты рождения.

*Подсказки (2): `dsc.hint('na-values')`*

In [ ]:
%%task ch03/02-missing/na-values
import pandas as pd
df = pd.read_csv("clients_raw.csv")
n_no_birth = df["birth_date"].isna().sum()

### Заказы с картой · база

В `orders_3m.csv` у части заказов пустой `client_id` — гость не показал карту. Сохраните в `card_pct` долю заказов **с картой** в процентах, округлённую до одного знака.

*Подсказки (2): `dsc.hint('card-share')`*

In [ ]:
%%task ch03/02-missing/card-share
import pandas as pd
orders = pd.read_csv("orders_3m.csv")

### Список для SMS · база

Маркетинг хочет разослать SMS. Сохраните в `sms` таблицу только с теми анкетами, где есть телефон, а в `n_lost` — сколько анкет не попало в рассылку.

*Подсказки (2): `dsc.hint('sms')`*

In [ ]:
%%task ch03/02-missing/sms
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Средний баланс: два ответа · база

Загрузите анкеты так, чтобы `bonus` стал числом (прочерк — пропуск, пробел — разделитель тысяч). Сохраните:

- `mean_known` — средний баланс по известным значениям;
- `mean_zero` — средний баланс, если пропуски считать нулями.

Обе цифры округлите до одного знака.

*Подсказки (2): `dsc.hint('bonus-mean')`*

In [ ]:
%%task ch03/02-missing/bonus-mean
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Сверка итогов по гостям · база

Маркетинг просит выручку по гостям из `orders_3m.csv`. Сгруппируйте заказы по `client_id` и сложите `amount` так, чтобы заказы без карты не потерялись, — Series `by_client`. Сохраните:

- `no_card` — **число**: выручку заказов без карты (значение группы-пропуска в `by_client`);
- `ok` — `True`, если сумма `by_client` совпадает с общей выручкой всех заказов. Получите его сравнением, а не впишите.

*Подсказки (3): `dsc.hint('ch03/02-missing/reconcile')`*

In [ ]:
%%task ch03/02-missing/reconcile
import pandas as pd
orders = pd.read_csv("orders_3m.csv")
by_client = orders.groupby("client_id")["amount"].sum()

### Почему пуст средний чек · база

В `guests_churn.csv` у части гостей пустой средний чек `avg_check`. Проверьте, связан ли пропуск с тем, установлено ли у гостя приложение кофейни (`app`: 1 — да, 0 — нет):

- `miss_by_app` — доля пропусков `avg_check` отдельно для гостей без приложения и с ним (Series с индексом `0` / `1`);
- `mechanism` — механизм пропуска строкой: `"MCAR"`, `"MAR"` или `"MNAR"`. Подсказка к выбору: доли в группах заметно различаются? Тогда смотрите в таблицу механизмов — от чего там зависит пропуск.

*Подсказки (2): `dsc.hint('mechanism')`*

In [ ]:
%%task ch03/02-missing/mechanism
import pandas as pd
guests = pd.read_csv("guests_churn.csv")

### Флаг перед заполнением · база

Загрузите анкеты с числовым `bonus` (как в прошлой задаче). Добавьте столбец `bonus_missing` (`True`, где баланс неизвестен), затем заполните пропуски в `bonus` нулями. Таблица — в `df`.

*Подсказки (2): `dsc.hint('flag')`*

In [ ]:
%%task ch03/02-missing/flag
import pandas as pd
df = pd.read_csv("clients_raw.csv", na_values=["—"], thousands=" ")

### Медиана своего города · посложнее

Заполните пропуски в `bonus` медианой бонусов гостей **из того же города** (город берите как есть, без очистки). Результат сохраните в столбец `bonus_filled`, а в `n_left` — сколько пропусков осталось после заполнения.

*Подсказки (2): `dsc.hint('group-fill')`*

In [ ]:
%%task ch03/02-missing/group-fill
import pandas as pd
df = pd.read_csv("clients_raw.csv", na_values=["—"], thousands=" ")

### Неполные анкеты · со звёздочкой

Сколько анкет, где пропущено **два поля или больше**? Замаскированные пропуски («не указано», «—») тоже считайте. Сохраните в `n_sparse`.

*Подсказки (2): `dsc.hint('two-plus')` · задача по желанию*

In [ ]:
%%task ch03/02-missing/two-plus
import pandas as pd
df = pd.read_csv("clients_raw.csv")

## Урок 3. Типы: из текста в числа и даты

Число, записанное как текст, нельзя сложить, а дату-строку нельзя сравнить с другой датой. Учимся читать выгрузки с нестандартным форматом, превращать текст в числа и даты и проверять, что при этом ничего не потерялось. Книги Excel с листами и шапкой — в следующем уроке.

Теория — на сайте курса: `ch03/03-types.html`. Задачи «собери код из строк» (1) остались на сайте.

### Прочитать кассу · разминка

Загрузите `pos_december.csv` в `pos` так, чтобы получилось 8 столбцов, а `price` стал числом. Сохраните в `gross` выручку без учёта скидок (сумма `price × qty`).

*Подсказки (2): `dsc.hint('read-pos')`*

In [ ]:
%%task ch03/03-types/read-pos
import pandas as pd
pos = pd.read_csv("pos_december.csv")

### Средняя оценка · разминка

Превратите `rating` в число (первый символ строки) и сохраните в `avg_stars` среднюю оценку, округлённую до двух знаков.

*Подсказки (2): `dsc.hint('stars')`*

In [ ]:
%%task ch03/03-types/stars
import pandas as pd
reviews = pd.read_csv("reviews.csv")

### Выручка со скидками · база

Превратите `discount` («10%», «0%», пусто) в долю: 10% → 0.1, пустая скидка → 0. Сохраните долю в столбец `disc`, а в `net` — выручку с учётом скидок (сумма `price × qty × (1 − disc)`), округлённую до двух знаков.

*Подсказки (2): `dsc.hint('ch03/03-types/discount')`*

In [ ]:
%%task ch03/03-types/discount
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")

### Бонусы без потерь · база

Превратите `bonus` в число через `pd.to_numeric(..., errors="coerce")` так, чтобы потерялся **только** прочерк. Сохраните результат в `bonus`, а в `lost` — **список** исходных непустых значений, которые не превратились в число (без повторов).

*Подсказки (2): `dsc.hint('bonus-num')`*

In [ ]:
%%task ch03/03-types/bonus-num
import pandas as pd
df = pd.read_csv("clients_raw.csv")
bonus = pd.to_numeric(df["bonus"], errors="coerce")

### Оценки — целые · база

Добавьте в `reviews` столбец `stars` — оценку целым числом с поддержкой пропусков (тип `Int64`). Сохраните в `n_no_stars` число отзывов без оценки.

*Подсказки (2): `dsc.hint('int64')`*

In [ ]:
%%task ch03/03-types/int64
import pandas as pd
reviews = pd.read_csv("reviews.csv")

### Время по кассе · база

Превратите `time` в дату-время и сохраните в `last_hour` **самый поздний час** (число от 0 до 23), в который пробивали заказы за всю неделю.

*Подсказки (2): `dsc.hint('shop-hours')`*

In [ ]:
%%task ch03/03-types/shop-hours
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")

### Дата рождения в двух форматах · посложнее

Превратите `birth_date` в дату: в столбце встречаются форматы «1990-05-12» и «12.05.1990». Сохраните результат в `born` (Series дат), а в `n_born` — сколько дат удалось разобрать.

*Подсказки (2): `dsc.hint('born')`*

In [ ]:
%%task ch03/03-types/born
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Возраст гостей · со звёздочкой

Посчитайте возраст гостей на 1 декабря 2026 года: число дней между датой рождения и этой датой, делённое на 365.25 и округлённое вниз до целого. Сохраните в `median_age` медианный возраст.

Подвох: среди дат есть те, что не могут быть датой рождения гостя кофейни. Их в расчёт не включайте — считайте только возрасты от 14 до 100 лет.

*Подсказки (2): `dsc.hint('ch03/03-types/age')` · задача по желанию*

In [ ]:
%%task ch03/03-types/age
import pandas as pd
df = pd.read_csv("clients_raw.csv")

## Урок 4. Книга Excel: листы, шапка, итоги

Финансовый отдел присылает не выгрузку, а книгу Excel, сделанную для людей: несколько листов, шапка над таблицей, строка «Итого», лишние столбцы. Читаем такую книгу так, чтобы числа стали числами, и сверяемся с её же итогом.

Теория — на сайте курса: `ch03/03b-excel.html`. Задачи «собери код из строк» (1) остались на сайте.

### Листы книги и «План» без шапки · разминка

В книге `zerno_finance.xlsx` несколько листов. Сохраните в `sheets` **список** имён всех листов книги — прочитайте её целиком через `sheet_name=None`. Затем прочитайте лист «План» в `plan` так, чтобы первым столбцом был «Код», а месяцы стали числами. В `n_shops` сохраните число кофеен — строк таблицы без «Итого».

*Подсказки (3): `dsc.hint('xl-skip')`*

In [ ]:
%%task ch03/03b-excel/xl-skip
import pandas as pd
plan = pd.read_excel("zerno_finance.xlsx")

### Год = сумма месяцев · база

В листе «План» книги `zerno_finance.xlsx` справа есть столбец «Год». Проверьте, что он действительно равен сумме двенадцати месяцев. Прочитайте лист через параметр `header=` (а не `skiprows`), уберите «Итого» и сделайте «Код» индексом. Сохраните:

- `year_sum` — сумму двенадцати месяцев по каждой кофейне (Series с кодами в индексе, один знак после запятой);
- `max_gap` — наибольшее по модулю расхождение между `year_sum` и столбцом «Год», один знак;
- `top` — код кофейни с самым большим годовым планом.

*Подсказки (2): `dsc.hint('year-check')`*

In [ ]:
%%task ch03/03b-excel/year-check
import pandas as pd
months = ["Янв", "Фев", "Мар", "Апр", "Май", "Июн", "Июл", "Авг", "Сен", "Окт", "Ноя", "Дек"]

### Сверка с итогом · база

Из листа «Факт» книги `zerno_finance.xlsx` прочитайте в `fact` **только** код кофейни и январь–март — параметром `usecols`. Шапка у этого листа — четыре строки. Сохраните:

- `q1` — выручку первого квартала по всем кофейням, посчитанную по строкам кофеен (без «Итого»);
- `q1_check` — то же число, но взятое из строки «Итого».

Оба числа округлите до одного знака.

*Подсказки (3): `dsc.hint('xl-total')`*

In [ ]:
%%task ch03/03b-excel/xl-total
import pandas as pd
fact = pd.read_excel("zerno_finance.xlsx", sheet_name="Факт")

### Кто отстаёт от плана · посложнее

План финансовый отдел прислал книгой `zerno_finance.xlsx` (лист «План»), а свежий факт учётная система выгрузила в CSV — `finance_fact.csv` (тот же лист «Факт», разделитель — точка с запятой). Сравните план и факт за январь–сентябрь. Сохраните в `done` выполнение плана каждой кофейней в процентах (факт / план × 100, один знак после запятой) — Series с кодами кофеен в индексе. Строку «Итого» и кофейню без плана за этот период в `done` не включайте. В `worst` сохраните код кофейни с самым низким выполнением.

*Подсказки (3): `dsc.hint('plan-fact')`*

In [ ]:
%%task ch03/03b-excel/plan-fact
import pandas as pd
months = ["Янв", "Фев", "Мар", "Апр", "Май", "Июн", "Июл", "Авг", "Сен"]

### Высота шапки кодом · со звёздочкой

В книге два листа с данными, и у каждого своя высота шапки. Пусть код сам найдёт строку с названиями столбцов — вдруг завтра финансисты добавят на лист ещё строку текста. Прочитайте листы «План» и «Факт» **без заголовка** (`header=None`) и для каждого найдите номер строки, где в первом столбце стоит «Код». Сохраните:

- `heads` — словарь «имя листа → номер этой строки» (считая с нуля, как в таблице, прочитанной с `header=None`);
- `pf` — таблицу с кодами кофеен в индексе (без «Итого») и двумя столбцами: `план` и `факт` — суммы за январь–сентябрь, один знак после запятой.

Числа `3` и `4` для шапок вписывать нельзя — найдите их.

*Подсказки (2): `dsc.hint('auto-header')` · задача по желанию*

In [ ]:
%%task ch03/03b-excel/auto-header
import pandas as pd
months = ["Янв", "Фев", "Мар", "Апр", "Май", "Июн", "Июл", "Авг", "Сен"]

## Урок 5. Дубликаты — явные и неявные

Одна и та же запись дважды — значит, выручка посчитана дважды, а гость учтён как двое. Ищем полные копии строк, повторы по ключу и неявные дубли, которые прячутся за разным написанием. И решаем, какую из копий оставить.

Теория — на сайте курса: `ch03/04-duplicates.html`. Задачи «собери код из строк» (1) остались на сайте.

### Полные копии · разминка

Сохраните в `n_exact` число полных дублей строк в анкетах.

*Подсказки (2): `dsc.hint('n-exact')`*

In [ ]:
%%task ch03/04-duplicates/n-exact
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Убрать копии · разминка

Сохраните в `clean` анкеты без полных дублей, с индексом по порядку от 0.

*Подсказки (2): `dsc.hint('drop-exact')`*

In [ ]:
%%task ch03/04-duplicates/drop-exact
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Нормализовать номер карты · база

Уберите полные дубли, затем приведите `client_id` к одному виду: без пробелов по краям, буквы — заглавные. Таблицу сохраните в `df`, а в `n_id_dups` — сколько после этого осталось повторов номера.

*Подсказки (2): `dsc.hint('norm-id')`*

In [ ]:
%%task ch03/04-duplicates/norm-id
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Самая свежая анкета · база

Оставьте по одной анкете на гостя — самую свежую по `signup_date`. Порядок: полные дубли → нормализация номера → сортировка → удаление повторов. Результат — в `clients` с индексом по порядку.

*Подсказки (2): `dsc.hint('keep-last')`*

In [ ]:
%%task ch03/04-duplicates/keep-last
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Какие заказы повторились · база

Сохраните в `dup_ids` отсортированный **список** номеров заказов (`id`), которые встречаются в кассовой выгрузке больше одного раза. Каждый номер — один раз.

*Подсказки (2): `dsc.hint('dup-ids')`*

In [ ]:
%%task ch03/04-duplicates/dup-ids
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")

### Прайс из заказов · база

В `orders_3m.csv` нет отдельного прайса, но цена чашки есть в каждом заказе: `amount / qty`. Соберите из заказов справочник цен `prices` — таблицу со столбцами `drink`, `size`, `price`, по одной строке на пару «напиток + размер». В `ok` сохраните `True`, если пара «напиток + размер» в `prices` действительно не повторяется (получите это проверкой, а не впишите).

*Подсказки (3): `dsc.hint('price-list')`*

In [ ]:
%%task ch03/04-duplicates/price-list
import pandas as pd
orders = pd.read_csv("orders_3m.csv")

### Цена дублей · посложнее

Насколько повторно отправленные заказы завышают выручку кассы? Сохраните в `overcount` разницу между суммой `price × qty` по сырой выгрузке и по выгрузке без повторов `id`, округлённую до двух знаков.

*Подсказки (2): `dsc.hint('overcount')`*

In [ ]:
%%task ch03/04-duplicates/overcount
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")

### Функция дедупликации · посложнее

Напишите функцию `dedupe(df, key, by)`: оставляет по одной строке на каждое значение столбца `key` — ту, у которой значение столбца `by` наибольшее (самая свежая дата, самая большая сумма). Индекс результата — по порядку от 0. Исходная таблица меняться не должна.

*Подсказки (2): `dsc.hint('dedupe-fn')`*

In [ ]:
%%task ch03/04-duplicates/dedupe-fn
import pandas as pd

def dedupe(df, key, by):
    ...

t = pd.DataFrame({"id": ["A", "B", "A"], "d": ["2026-01-05", "2026-02-01", "2026-03-10"], "v": [1, 2, 3]})
dedupe(t, "id", "d")

### Справочник кофеен · посложнее

Соберите из `delivery_flat.csv` справочник кофеен `shops` — по одной строке на `shop_id` со столбцами `shop_id`, `shop_address`, `city`, `manager`:

- город приведите к одному виду: без пробелов по краям, с заглавной буквы;
- если у кофейни сменился управляющий, оставьте самого свежего (по дате заказа).

В `n_changed` сохраните, у скольких кофеен **после нормализации города** осталось больше одной версии атрибутов.

*Подсказки (3): `dsc.hint('shops-ref')`*

In [ ]:
%%task ch03/04-duplicates/shops-ref
import pandas as pd
flat = pd.read_csv("delivery_flat.csv")
cols = ["shop_id", "shop_address", "city", "manager"]
shops = flat[cols].drop_duplicates()

### Все версии анкет · со звёздочкой

Для проверки маркетинг просит показать все анкеты гостей, которые заполняли её больше одного раза. Уберите полные дубли, нормализуйте номер карты и сохраните в `versions` **все** строки с повторяющимся номером, отсортированные по номеру и дате анкеты. В `n_guests` — сколько таких гостей.

*Подсказки (2): `dsc.hint('ch03/04-duplicates/versions')` · задача по желанию*

In [ ]:
%%task ch03/04-duplicates/versions
import pandas as pd
df = pd.read_csv("clients_raw.csv")

## Урок 6. Строки: чистим и ищем по тексту

Большая часть грязи — в тексте: регистр, пробелы, приставки вроде «г.», синонимы и опечатки, телефоны в пяти форматах. Учимся приводить текст к одному виду методами .str, заменять по словарю и искать по словам — и немного регулярных выражений, без которых в чистке не обойтись.

Теория — на сайте курса: `ch03/05-strings.html`. Задачи «собери код из строк» (1) остались на сайте.

### Регистр и пробелы · разминка

Сохраните в `n_after` число разных значений города после `strip()` и `lower()` (без пропусков). Сравните с тем, что было до.

*Подсказки (2): `dsc.hint('city-lower')`*

In [ ]:
%%task ch03/05-strings/city-lower
import pandas as pd
df = pd.read_csv("clients_raw.csv")
print("до:", df["city"].nunique())

### Холодный кофе · разминка

Сохраните в `n_cold` число отзывов, где упоминается холодный или остывший напиток: ищите «холодн» **или** «остыл», без учёта регистра.

*Подсказки (2): `dsc.hint('cold')`*

In [ ]:
%%task ch03/05-strings/cold
import pandas as pd
reviews = pd.read_csv("reviews.csv")

### Город начисто · база

Приведите город к каноническому виду: «Пермь», «Казань», «Тюмень», «Уфа», «Екатеринбург», «Москва» (пропуски оставьте пропусками). Сохраните результат в столбец `city` таблицы `df`.

*Подсказки (2): `dsc.hint('city-clean')`*

In [ ]:
%%task ch03/05-strings/city-clean
import pandas as pd
df = pd.read_csv("clients_raw.csv")

df["city"].value_counts(dropna=False)

### Телефон в одном формате · база

Сохраните в столбец `phone10` телефон из 10 цифр (без кода страны). Пустые телефоны оставьте пропусками. В `n_bad` — число непустых `phone10`, у которых длина не 10 (должно быть 0).

*Подсказки (2): `dsc.hint('phone10')`*

In [ ]:
%%task ch03/05-strings/phone10
import pandas as pd
df = pd.read_csv("clients_raw.csv")

### Напитки по меню · база

Приведите названия напитков в кассовой выгрузке к названиям из меню: пробелы, регистр и три опечатки («Каппучино», «Флет уайт», «Амерекано»). Сохраните результат в `pos["drink"]`, а в `unknown` — множество названий, которых нет в меню (должно получиться пустое).

*Подсказки (2): `dsc.hint('ch03/05-strings/drinks')`*

In [ ]:
%%task ch03/05-strings/drinks
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")
menu = pd.read_csv("menu.csv")

### Код кофейни · база

В кассовой выгрузке код кофейни записан как «PRM-1», «prm-1» или «PRM 1». Приведите `shop` к виду «PRM-1» и сохраните в `n_shops` число разных кофеен после чистки.

*Подсказки (2): `dsc.hint('shop-code')`*

In [ ]:
%%task ch03/05-strings/shop-code
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")

### Темы жалоб · посложнее

Посчитайте, сколько отзывов относится к каждой теме, и сохраните словарь `themes`:

- `"холодно"` — «холодн» или «остыл»;
- `"долго"` — «долг», «очеред» или «ждал»;
- `"дорого"` — «дорог».

Регистр не важен. Один отзыв может попасть в несколько тем.

*Подсказки (2): `dsc.hint('themes')`*

In [ ]:
%%task ch03/05-strings/themes
import pandas as pd
reviews = pd.read_csv("reviews.csv")

### Один телефон — разные карты · со звёздочкой

Некоторые гости завели новую карту, не закрыв старую. Найдите телефоны (в виде 10 цифр), на которые записано **больше одной** разной карты. Номер карты предварительно нормализуйте (пробелы, регистр). Сохраните в `shared` отсортированный список таких телефонов.

*Подсказки (2): `dsc.hint('shared-phone')` · задача по желанию*

In [ ]:
%%task ch03/05-strings/shared-phone
import pandas as pd
df = pd.read_csv("clients_raw.csv")

## Урок 7. Категоризация: CASE WHEN в pandas

Отчёты часто строят не по точным значениям, а по группам: возрастные группы, «маленький — средний — крупный» заказ, редкие и частые гости. Учимся раскладывать числа и текст по категориям — функцией, интервалами и словарём — и собирать редкие значения в «прочее».

Теория — на сайте курса: `ch03/06-categories.html`. Задачи «собери код из строк» (1) остались на сайте.

### Размер заказа · разминка

Напишите функцию `check_size(amount)`: до 300 ₽ — «малый», от 300 до 600 — «средний», от 600 — «крупный». Добавьте столбец `check` через `apply` и сохраните в `sizes` словарь «категория → число заказов».

*Подсказки (2): `dsc.hint('check-size')`*

In [ ]:
%%task ch03/06-categories/check-size
import pandas as pd
df = pd.read_csv("orders_3m.csv")

def check_size(amount):
    ...

### Время суток · разминка

Разбейте заказы по времени суток через `pd.cut`: утро — с 0 до 12 часов, день — с 12 до 17, вечер — с 17 до 24 (левая граница включается). Сохраните в `evening` число вечерних заказов.

*Подсказки (2): `dsc.hint('parts')`*

In [ ]:
%%task ch03/06-categories/parts
import pandas as pd
df = pd.read_csv("orders_3m.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

### Частота визитов · база

Посчитайте число заказов у каждого гостя с картой и разложите гостей по группам через `pd.cut`: 1–3 заказа — «редкие», 4–6 — «обычные», 7 и больше — «частые». Сохраните в `segments` словарь «группа → число гостей».

*Подсказки (2): `dsc.hint('freq')`*

In [ ]:
%%task ch03/06-categories/freq
import pandas as pd
df = pd.read_csv("orders_3m.csv")

### Верхняя четверть · база

Разбейте гостей с картой на 4 равные группы по сумме покупок (`pd.qcut`) и сохраните в `top_share` долю выручки верхней четверти (в процентах от выручки всех гостей с картой, один знак).

*Подсказки (2): `dsc.hint('ch03/06-categories/quartiles')`*

In [ ]:
%%task ch03/06-categories/quartiles
import pandas as pd
df = pd.read_csv("orders_3m.csv")

### Онлайн или офлайн · база

Сведите источник к двум группам. Сначала нормализуйте (`strip`, `lower`) и примените словарь синонимов из урока, затем укрупните: «соцсети», «карты», «реклама» → «онлайн»; «друзья», «мимо» → «офлайн». Остальные значения и пропуски — «другое». Результат — в столбец `channel`, в `channels` — словарь «группа → число анкет».

*Подсказки (2): `dsc.hint('online')`*

In [ ]:
%%task ch03/06-categories/online
import pandas as pd
df = pd.read_csv("clients_raw.csv")
synonyms = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

### Собрать редкие · посложнее

Напишите функцию `lump(s, min_count)`: значения, которые встречаются меньше `min_count` раз, заменяются на «прочее»; частые значения и пропуски остаются как есть. Сохраните в `src` очищенный источник (нормализация + синонимы) после `lump(..., 10)`.

*Подсказки (2): `dsc.hint('ch03/06-categories/rare')`*

In [ ]:
%%task ch03/06-categories/rare
import pandas as pd
df = pd.read_csv("clients_raw.csv")
synonyms = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

def lump(s, min_count):
    ...

src = lump(df["source"].str.strip().str.lower().replace(synonyms), 10)
src.value_counts(dropna=False)

### Возрастные группы · посложнее

Посчитайте возраст гостей на 1 декабря 2026 года, как в уроке «Типы: из текста в числа и даты» (дата рождения в двух форматах, `дни // 365.25`). Разложите по группам через `pd.cut` с `right=False`: 14–24, 25–34, 35–44, 45–59, 60–99 — с подписями `"14–24"`, `"25–34"` и т. д. (можно писать через обычный дефис: `"14-24"`, проверка примет оба варианта). Возрасты вне этих интервалов станут пропусками. Сохраните в `groups` словарь «группа → число анкет».

*Подсказки (2): `dsc.hint('age-groups')`*

In [ ]:
%%task ch03/06-categories/age-groups
import pandas as pd
df = pd.read_csv("clients_raw.csv")
iso = pd.to_datetime(df["birth_date"], format="%Y-%m-%d", errors="coerce")
ru = pd.to_datetime(df["birth_date"], format="%d.%m.%Y", errors="coerce")
born = iso.fillna(ru)

### Сегменты гостей · со звёздочкой

Для каждого гостя с картой посчитайте `orders` (число заказов) и `avg_check` (средний чек). Присвойте сегмент функцией от строки:

- «VIP» — 7 и больше заказов **и** средний чек не ниже медианы средних чеков всех гостей;
- «частый» — 7 и больше заказов, чек ниже медианы;
- «щедрый» — меньше 7 заказов, чек не ниже медианы;
- «обычный» — остальные.

Таблицу гостей с столбцом `segment` сохраните в `guests`, а в `n_vip` — число VIP.

*Подсказки (2): `dsc.hint('vip')` · задача по желанию*

In [ ]:
%%task ch03/06-categories/vip
import pandas as pd
df = pd.read_csv("orders_3m.csv")

## Урок 8. Аномалии и проверка на здравый смысл

Данные могут быть правильного типа, без пропусков и дублей — и всё равно неверными: отрицательный баланс, гость 1900 года рождения, заказ из нуля чашек. Учимся находить такие значения правилами здравого смысла и статистикой и решать, что с ними делать.

Теория — на сайте курса: `ch03/07-anomalies.html`. Задачи «собери код из строк» (1) остались на сайте.

### Отрицательные бонусы · разминка

Бонусы уже превращены в число. Сохраните в `n_negative` число анкет с отрицательным балансом.

*Подсказки (2): `dsc.hint('negative')`*

In [ ]:
%%task ch03/07-anomalies/negative
import pandas as pd
df = pd.read_csv("clients_raw.csv")
df["bonus"] = pd.to_numeric(df["bonus"].str.replace(" ", ""), errors="coerce")

### Пустые заказы · разминка

Сохраните в `zero_ids` **список** номеров заказов (`id`) из кассы, где количество равно нулю.

*Подсказки (2): `dsc.hint('zero-qty')`*

In [ ]:
%%task ch03/07-anomalies/zero-qty
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")

### Заглушки в датах · база

Разберите дату рождения (оба формата, как в уроке «Типы: из текста в числа и даты»). Сохраните:

- `n_1900` — сколько анкет с заглушкой 1900-01-01;
- `n_future` — сколько дат рождения позже 1 декабря 2026 года.

*Подсказки (2): `dsc.hint('placeholder')`*

In [ ]:
%%task ch03/07-anomalies/placeholder
import pandas as pd
df = pd.read_csv("clients_raw.csv")
iso = pd.to_datetime(df["birth_date"], format="%Y-%m-%d", errors="coerce")
ru = pd.to_datetime(df["birth_date"], format="%d.%m.%Y", errors="coerce")
born = iso.fillna(ru)

### Бонусы без аномалий · база

Замените на `NaN` невозможные балансы: отрицательные и больше 10 000 (это заглушки). Используйте `mask`. Сохраните очищенный столбец в `df["bonus"]`, а в `median_before` и `mean_after` — медиану **до** очистки и среднее **после** (оба округлите до 1 знака).

*Подсказки (2): `dsc.hint('mask-bonus')`*

In [ ]:
%%task ch03/07-anomalies/mask-bonus
import pandas as pd
df = pd.read_csv("clients_raw.csv")
df["bonus"] = pd.to_numeric(df["bonus"].str.replace(" ", ""), errors="coerce")

### Тикет про заглушку в датах · база

В уроке нашлась заглушка 1900-01-01 в дате рождения. Соберите тикет для владельца CRM — словарь `report` с ключами:

- `"где"` — файл и столбец (строка);
- `"правило"` — какое правило нарушено (строка);
- `"строк"` — сколько анкет с заглушкой;
- `"всего"` — сколько анкет в выгрузке;
- `"доля_%"` — доля анкет с заглушкой в процентах, один знак после запятой;
- `"примеры"` — список номеров карт (`client_id`) таких анкет, не больше пяти, в порядке файла;
- `"как_найти"` — условие, по которому владелец найдёт эти строки (строка).

Все числа и примеры посчитайте кодом. Имена и телефоны гостей в тикет не кладите.

*Подсказки (3): `dsc.hint('bug-report')`*

In [ ]:
%%task ch03/07-anomalies/bug-report
import pandas as pd
df = pd.read_csv("clients_raw.csv")
iso = pd.to_datetime(df["birth_date"], format="%Y-%m-%d", errors="coerce")
ru = pd.to_datetime(df["birth_date"], format="%d.%m.%Y", errors="coerce")
born = iso.fillna(ru)

report = {}

### Границы выбросов · база

Посчитайте по правилу межквартильного размаха границы `low` и `high` для суммы заказа (`amount` в `orders_3m.csv`) и сохраните в `n_high` число заказов выше верхней границы.

*Подсказки (2): `dsc.hint('iqr')`*

In [ ]:
%%task ch03/07-anomalies/iqr
import pandas as pd
df = pd.read_csv("orders_3m.csv")

### Флаг подозрительных заказов · база

Добавьте в кассовую выгрузку столбец-флаг `suspicious`: `True`, если количество 0 или больше 10. Строки не удаляйте. Сохраните в `avg_with` и `avg_without` среднюю сумму **заказа** (`price × qty`) со всеми строками и без подозрительных (обе округлите до 2 знаков).

*Подсказки (2): `dsc.hint('flag-qty')`*

In [ ]:
%%task ch03/07-anomalies/flag-qty
import pandas as pd
pos = pd.read_csv("pos_december.csv", sep=";", decimal=",")

### Правдоподобный возраст · посложнее

Посчитайте возраст на 1 декабря 2026 (`дни // 365.25`) и замените на `NaN` возрасты вне диапазона 14–100 лет (через `mask` или `where`). Сохраните возраст в `age`, а в `n_valid` — число анкет с правдоподобным возрастом.

*Подсказки (2): `dsc.hint('age-sane')`*

In [ ]:
%%task ch03/07-anomalies/age-sane
import pandas as pd
df = pd.read_csv("clients_raw.csv")
iso = pd.to_datetime(df["birth_date"], format="%Y-%m-%d", errors="coerce")
ru = pd.to_datetime(df["birth_date"], format="%d.%m.%Y", errors="coerce")
born = iso.fillna(ru)

### Анкета раньше кофейни · со звёздочкой

Сколько гостей заполнили анкету **раньше**, чем в их городе открылась первая кофейня «Зерна»? Город очистите как в уроке «Строки: чистим и ищем по тексту», дату открытия первой кофейни города возьмите из `shops.csv` (самую раннюю по городу). Гости из городов без кофеен в расчёт не входят. Сохраните ответ в `n_early`, а в `by_city` — словарь «город → число таких анкет» (только города, где они есть).

*Подсказки (2): `dsc.hint('before-open')` · задача по желанию*

In [ ]:
%%task ch03/07-anomalies/before-open
import pandas as pd
df = pd.read_csv("clients_raw.csv")
shops = pd.read_csv("shops.csv")
df["city"] = (df["city"].str.strip().str.lower().str.removeprefix("г.").str.strip()
              .replace({"екб": "екатеринбург"}).str.capitalize())

## Урок 9. Очистка целиком: функция и проверки

Через месяц маркетинг пришлёт новую выгрузку анкет — и всю очистку придётся повторять. Собираем шаги прошлых уроков в функции, которые запускаются одной строкой, ведём журнал потерь, проверяем результат автоматически и сохраняем чистые данные.

Теория — на сайте курса: `ch03/08-pipeline.html`. Задачи «собери код из строк» (1) остались на сайте.

### Функция для города · разминка

Напишите `clean_city(s)`: получает Series с городами и возвращает очищенную (как в уроке «Строки: чистим и ищем по тексту»).

*Подсказки (2): `dsc.hint('fn-city')`*

In [ ]:
%%task ch03/08-pipeline/fn-city
import pandas as pd

def clean_city(s):
    ...

clean_city(pd.Series([" г. Пермь", "ЕКБ", "казань ", None]))

### Функция для пола · разминка

Напишите `clean_gender(s)`: «М», «м», «муж» → «М»; «Ж», «ж», «жен» → «Ж»; пропуски и всё прочее → пропуск.

*Подсказки (2): `dsc.hint('fn-gender')`*

In [ ]:
%%task ch03/08-pipeline/fn-gender
import pandas as pd

def clean_gender(s):
    ...

clean_gender(pd.Series(["М", "муж", "ж", None, "жен "]))

### Функция для бонусов · база

Напишите `clean_bonus(s)`: убирает пробелы, превращает в число (не-числа → `NaN`) и заменяет на `NaN` значения меньше 0 и больше 10 000.

*Подсказки (2): `dsc.hint('fn-bonus')`*

In [ ]:
%%task ch03/08-pipeline/fn-bonus
import pandas as pd

def clean_bonus(s):
    ...

clean_bonus(pd.Series(["1 250", "—", "-50", "99999", "320", None]))

### Почините присваивание · база

Коллега хотел исправить бонусы: отрицательные заменить на 0, а заглушки больше 10 000 — на пропуск. Код выполняется, выводит предупреждения, но таблица не меняется. Исправьте его так, чтобы менялась сама `df`. В `n_fixed` сохраните, сколько значений бонусов изменилось (посчитайте до исправления).

*Подсказки (3): `dsc.hint('fix-chain')`*

In [ ]:
%%task ch03/08-pipeline/fix-chain
import pandas as pd
df = pd.read_csv("clients_raw.csv")
df["bonus"] = pd.to_numeric(df["bonus"].str.replace(" ", ""), errors="coerce")

df[df["bonus"] < 0]["bonus"] = 0
df[df["bonus"] > 10_000]["bonus"] = None
n_fixed = 0

### Сохранить и прочитать · база

Таблица `clients` уже очищена. Сохраните её в `clients_clean.csv` без индекса и прочитайте обратно в `back` так, чтобы `signup_date` снова был датой.

*Подсказки (2): `dsc.hint('save')`*

In [ ]:
%%task ch03/08-pipeline/save
import pandas as pd

### Очистка с журналом · база

Напишите `clean_with_log(raw)`, которая возвращает пару `(df, log)`: `df` — анкеты после трёх шагов (полные дубли → нормализация номера → самая свежая анкета), `log` — список пар `(название шага, число строк после шага)`, начиная с `("загрузка", len(raw))`. Всего в журнале 4 записи.

*Подсказки (2): `dsc.hint('ch03/08-pipeline/log')`*

In [ ]:
%%task ch03/08-pipeline/log
import pandas as pd

def clean_with_log(raw):
    log = [("загрузка", len(raw))]
    ...

df, log = clean_with_log(pd.read_csv("clients_raw.csv"))
log

### Контракт анкет · посложнее

Напишите `check_clients(df)` с проверками через `assert`:

- `client_id` без пропусков и уникален;
- `gender` — только «М», «Ж» или пропуск;
- `city` — только «Пермь», «Казань», «Тюмень», «Уфа», «Екатеринбург», «Москва» или пропуск;
- `bonus` — от 0 до 10 000 или пропуск.

Если всё в порядке, функция ничего не возвращает; если нет — падает с `AssertionError`.

*Подсказки (2): `dsc.hint('ch03/08-pipeline/contract')`*

In [ ]:
%%task ch03/08-pipeline/contract
import pandas as pd

def check_clients(df):
    ...

ok = pd.DataFrame({"client_id": ["C1", "C2"], "gender": ["М", None],
                   "city": ["Пермь", None], "bonus": [100.0, None]})
check_clients(ok)
print("контракт выполнен")

### Вся очистка одной функцией · посложнее

Соберите `clean_clients(raw)` — полную очистку анкет. Функции для отдельных столбцов уже написаны в заготовке. Функция должна:

1. убрать полные дубли;
2. нормализовать `client_id` (пробелы, регистр), `name` (пробелы, первая буква заглавная);
3. почистить `gender`, `birth_date`, `city`, `phone`, `bonus` готовыми функциями; `signup_date` превратить в дату; `source` — `strip`, `lower` и словарь `SYNONYMS`;
4. оставить по одной, самой свежей анкете на гостя; индекс — по порядку.

Сохраните результат для `clients_raw.csv` в `clients`.

*Подсказки (2): `dsc.hint('full')`*

In [ ]:
%%task ch03/08-pipeline/full
import pandas as pd

SYNONYMS = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

def clean_city(s):
    return (s.str.strip().str.lower().str.removeprefix("г.").str.strip()
             .replace({"екб": "екатеринбург"}).str.capitalize())

def clean_phone(s):
    return s.str.replace(r"\D", "", regex=True).str[-10:]

def clean_gender(s):
    return s.str.strip().str.lower().str[0].map({"м": "М", "ж": "Ж"})

def parse_birth(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    ru = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    born = iso.fillna(ru)
    return born.where(born.between("1926-12-01", "2012-12-01"))

def clean_bonus(s):
    b = pd.to_numeric(s.str.replace(" ", ""), errors="coerce")
    return b.mask((b < 0) | (b > 10_000))

def clean_clients(raw):
    ...

clients = clean_clients(pd.read_csv("clients_raw.csv"))
clients.head()

### Новая выгрузка · со звёздочкой

Пришла январская выгрузка `clients_raw_jan.csv`. В ней появились новые написания города: «гор. Уфа» и «Пермь, Россия». Доработайте `clean_city`, чтобы она справлялась и с ними (и по-прежнему со старыми вариантами), и сохраните в `jan_cities` множество городов январской выгрузки после очистки (без пропусков).

*Подсказки (2): `dsc.hint('january')` · задача по желанию*

In [ ]:
%%task ch03/08-pipeline/january
import pandas as pd

def clean_city(s):
    return (s.str.strip().str.lower().str.removeprefix("г.").str.strip()
             .replace({"екб": "екатеринбург"}).str.capitalize())

jan = pd.read_csv("clients_raw_jan.csv")
jan_cities = set(clean_city(jan["city"]).dropna())
jan_cities

## Урок 10. Проект: кто наши гости

Финал главы. Маркетинг хочет понять гостей программы лояльности: сколько их на самом деле, кто они по возрасту и городу, откуда о нас узнают и кто приносит выручку. Данные — грязные анкеты из старой CRM и заказы за осень.

Теория — на сайте курса: `ch03/09-project.html`.

### Шаг 1. Очистка анкет · база

Очистите анкеты так же, как в уроке «Очистка целиком: функция и проверки»: функции для столбцов уже в заготовке, допишите `clean_clients(raw)` (или перенесите свою из урока «Очистка целиком: функция и проверки»). Результат — `clients`. Заодно соберите `log` — словарь с тремя числами:

- `"выгрузка"` — строк в сыром файле;
- `"без полных дублей"` — после удаления полных копий;
- `"анкет"` — строк в `clients`.

*Подсказки (2): `dsc.hint('ch03/09-project/clean')`*

In [ ]:
%%task ch03/09-project/clean
import pandas as pd

SYNONYMS = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

def clean_city(s):
    return (s.str.strip().str.lower().str.removeprefix("г.").str.strip()
             .replace({"екб": "екатеринбург"}).str.capitalize())

def clean_phone(s):
    return s.str.replace(r"\D", "", regex=True).str[-10:]

def clean_gender(s):
    return s.str.strip().str.lower().str[0].map({"м": "М", "ж": "Ж"})

def parse_birth(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    ru = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    born = iso.fillna(ru)
    return born.where(born.between("1926-12-01", "2012-12-01"))

def clean_bonus(s):
    b = pd.to_numeric(s.str.replace(" ", ""), errors="coerce")
    return b.mask((b < 0) | (b > 10_000))

def clean_clients(raw):
    ...

raw = pd.read_csv("clients_raw.csv")
clients = clean_clients(raw)
log = {}
log

### Шаг 2. Качество после очистки · база

Таблица `clients` уже очищена. Сохраните в `missing_pct` долю пропусков по столбцам в процентах (1 знак), отсортированную по убыванию, и в `usable_age` — долю анкет, по которым известен возраст (то есть есть `birth_date`), в процентах с одним знаком.

*Подсказки (2): `dsc.hint('ch03/09-project/quality')`*

In [ ]:
%%task ch03/09-project/quality
import pandas as pd
# clients — очищенные анкеты, orders — заказы за осень (datetime — дата)
clients.head(3)

### Шаг 3. Портрет по возрасту и городам · база

Добавьте в `clients` столбцы `age` (полных лет на 1 декабря 2026: `дни // 365.25`) и `age_group` (`pd.cut` с `right=False`, границы 14, 25, 35, 45, 60, 100 и подписи `"14–24"`, `"25–34"`, `"35–44"`, `"45–59"`, `"60–99"`). Сохраните:

- `median_age` — медианный возраст;
- `age_counts` — словарь «группа → число анкет»;
- `by_city` — словарь «город → число анкет» (без пропусков).

*Подсказки (2): `dsc.hint('ch03/09-project/age')`*

In [ ]:
%%task ch03/09-project/age
import pandas as pd

### Шаг 4. Анкеты и заказы · база

Соедините анкеты с заказами. Посчитайте по каждому гостю число заказов `orders` и выручку `revenue` и присоедините к `clients` (LEFT JOIN — гости без заказов остаются, у них `orders` и `revenue` равны 0). Результат — `guests`. Сохраните:

- `active_pct` — долю гостей хотя бы с одним заказом, %, 1 знак;
- `n_orphans` — сколько разных номеров карт есть в заказах, но нет в анкетах (проверка между таблицами). Ноль — тоже возможный ответ: значит, проверка ничего не нашла.

*Подсказки (3): `dsc.hint('guests')`*

In [ ]:
%%task ch03/09-project/guests
import pandas as pd

### Шаг 5. Откуда приходят гости · база

В `guests` уже есть заказы и выручка гостя. Подготовьте источник: пропуски → «не указано», затем редкие значения (меньше 10 анкет) → «прочее» (функция `lump` из урока «Категоризация: CASE WHEN в pandas»). Соберите `by_source` — таблицу по источникам со столбцами:

- `clients` — число анкет;
- `active_pct` — доля гостей с заказами, % (1 знак);
- `avg_revenue` — средняя выручка на гостя за осень, включая гостей без заказов, округлённая до целого рубля: `round(0)` (подойдёт и дробная запись вида `702.0`, и `astype(int)` после округления).

Отсортируйте по `clients` по убыванию. В `best_source` сохраните источник с наибольшей `avg_revenue` среди групп, где **не меньше 20 гостей**.

*Подсказки (2): `dsc.hint('sources')`*

In [ ]:
%%task ch03/09-project/sources
import pandas as pd
# guests — анкеты + orders и revenue; source — после словаря синонимов
guests["source"].value_counts(dropna=False)

### Шаг 6. Возраст и деньги · посложнее

Среди **активных** гостей (хотя бы один заказ) с известной возрастной группой посчитайте по группам `age_group`: число гостей `guests_n`, среднее число заказов `avg_orders` и среднюю выручку `avg_revenue` (оба — 1 знак). Таблица — `by_age`, в `best_group` — группа с наибольшей средней выручкой.

*Подсказки (2): `dsc.hint('age-money')`*

In [ ]:
%%task ch03/09-project/age-money
import pandas as pd
# в guests уже есть age, age_group, orders, revenue
guests[["client_id", "age", "age_group", "orders", "revenue"]].head()

### Шаг 7. Свой город или в гостях · посложнее

Гость из Перми может заказать кофе и в Казани. Для заказов с картой определите город кофейни (из `shops.csv`) и город гостя (из анкеты). Среди заказов, где известны оба города, посчитайте `home_pct` — долю заказов в городе гостя (%, 1 знак). В `travelers` сохраните число гостей, которые заказывали в двух и более городах.

*Подсказки (3): `dsc.hint('home')`*

In [ ]:
%%task ch03/09-project/home
import pandas as pd
shops = pd.read_csv("shops.csv")

### Шаг 8. Отчёт одной функцией · со звёздочкой

Напишите `report(raw, orders)`: принимает **сырые** анкеты и заказы, очищает анкеты (функцией `clean_clients` из шага 1, она уже в заготовке) и возвращает словарь:

- `"clients"` — число анкет после очистки;
- `"active_pct"` — доля гостей хотя бы с одним заказом, % (1 знак);
- `"median_age"` — медианный возраст на 1 декабря 2026;
- `"avg_revenue_active"` — средняя выручка активного гостя (1 знак).

Функция должна работать на любом периоде заказов — например, только на ноябре.

*Подсказки (2): `dsc.hint('ch03/09-project/report')` · задача по желанию*

In [ ]:
%%task ch03/09-project/report
import pandas as pd

SYNONYMS = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

def clean_city(s):
    return (s.str.strip().str.lower().str.removeprefix("г.").str.strip()
             .replace({"екб": "екатеринбург"}).str.capitalize())

def clean_phone(s):
    return s.str.replace(r"\D", "", regex=True).str[-10:]

def clean_gender(s):
    return s.str.strip().str.lower().str[0].map({"м": "М", "ж": "Ж"})

def parse_birth(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    ru = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    born = iso.fillna(ru)
    return born.where(born.between("1926-12-01", "2012-12-01"))

def clean_bonus(s):
    b = pd.to_numeric(s.str.replace(" ", ""), errors="coerce")
    return b.mask((b < 0) | (b > 10_000))

def clean_clients(raw):
    df = raw.drop_duplicates().copy()
    df["client_id"] = df["client_id"].str.strip().str.upper()
    df["name"] = df["name"].str.strip().str.capitalize()
    df["gender"] = clean_gender(df["gender"])
    df["birth_date"] = parse_birth(df["birth_date"])
    df["city"] = clean_city(df["city"])
    df["phone"] = clean_phone(df["phone"])
    df["signup_date"] = pd.to_datetime(df["signup_date"])
    df["bonus"] = clean_bonus(df["bonus"])
    df["source"] = df["source"].str.strip().str.lower().replace(SYNONYMS)
    return (df.sort_values("signup_date")
              .drop_duplicates(subset=["client_id"], keep="last")
              .reset_index(drop=True))

def report(raw, orders):
    ...

raw = pd.read_csv("clients_raw.csv")
orders = pd.read_csv("orders_3m.csv", parse_dates=["datetime"])
report(raw, orders)

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()